# 08 · The framework outside aviation: stocking under uncertain demand (newsvendor)

The tutor's point: the product is the decision framework, not the jet-engine model. Retail has the same structure (stockout ↔ failure, overstock ↔ early removal), and the cost-optimal stock is the τ-quantile of demand with τ = ratio / (1 + ratio).

**Data.** `results/retail_synthetic_decisions.csv` from `python -m src.run retail --data synthetic --seeds 0-4` (12 synthetic store series with weekly seasonality, trend and demand shocks; runs anywhere). With the Kaggle M5 files in `data/m5/` (see `src/m5.py`), the same command with `--data m5` runs on the 70 store × department Walmart series.

**Split by time.** Train ≤ day 1857, calibration 1858–1885, test 1886–1913, 28-day-ahead direct forecasts (lags ≥ 28).

In [1]:
import pandas as pd
from pathlib import Path
from src.config import RESULTS_DIR
from src.decision import break_even

pd.set_option("display.width", 180)
files = sorted(Path(RESULTS_DIR).glob("retail_*_decisions.csv"))
ret = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
ret.groupby("data")[["seed"]].nunique()

,seed
data,
synthetic,5


## Cost per day (units of one overstocked item), median over seeds

In [2]:
for data, d in ret.groupby("data"):
    print(data); display(d.pivot_table(index="rule", columns="ratio", values="cost_per_period", aggfunc="median").round(1))

synthetic


ratio,1.0,1.5,2.0,3.0,5.0,10.0,20.0,50.0,100.0
rule,,,,,,,,,
conformal tau,222.7,275.7,318.1,394.4,503.1,671.3,853.8,1119.9,1397.9
point,223.5,280.4,337.9,452.9,682.8,1257.7,2407.5,5856.8,11605.7
point + tuned buffer,223.5,275.4,318.3,392.2,502.2,671.2,851.6,1109.1,1351.8
quantile tau,224.2,276.0,320.4,384.5,473.1,581.1,702.8,905.2,1156.4


## Stockout share

In [3]:
ret.pivot_table(index=["data", "rule"], columns="ratio", values="stockout_share", aggfunc="median").round(3)

ratio                           1.0    1.5    2.0    3.0    5.0    10.0   20.0   50.0   100.0
data      rule                                                                               
synthetic conformal tau         0.473  0.357  0.295  0.214  0.140  0.086  0.060  0.015  0.003
          point                 0.467  0.467  0.467  0.467  0.467  0.467  0.467  0.467  0.467
          point + tuned buffer  0.435  0.387  0.304  0.211  0.140  0.089  0.062  0.021  0.006
          quantile tau          0.464  0.363  0.301  0.226  0.158  0.086  0.048  0.024  0.006

## Break-even against the tuned buffer (the retail twin of the tuned safety margin)

In [4]:
rows = []
for (data, seed), d in ret.groupby(["data", "seed"]):
    t = d.rename(columns={"cost_per_period": "cost"})
    for c in ["conformal tau", "quantile tau"]:
        rows.append({"data": data, "seed": seed, "challenger": c, "break_even": break_even(t, "point + tuned buffer", c, metric="cost")})
pd.DataFrame(rows).groupby(["data", "challenger"])["break_even"].describe()[["count", "50%", "min", "max"]]

count        50%        min        max
data      challenger                                           
synthetic conformal tau    2.0  17.153063  13.417064  20.889063
          quantile tau     5.0   1.041091   0.000000   2.214451

## Reading

Synthetic run, 5 seeds (replace with `--data m5` once the Kaggle files are in place):

* **The point forecast** is only right when a stockout and an overstock cost the same; at 10 : 1 it costs almost twice the alternatives.
* **A constant conformal shift is a tuned buffer in disguise.** It behaves like the buffer at every ratio and only breaks even in 2 of 5 seeds, at 13-21 : 1. This is the retail twin of the engine result "split conformal = point prediction + safety margin".
* **The adaptive quantile forecast (conformalised) wins from about 1-2 : 1** (median break-even 1.04, range 0-2.2) and is 13 % cheaper than the tuned buffer at 10 : 1 (581 vs 671 per day).

The break-even ratio is the transferable output: one number a planner in any sector can set against their own costs.